# Kinematic plots for different networks

In [ ]:
import sys
import json
import matplotlib.pyplot as plt
import numpy as np
import argparse
import re

from pathlib import Path
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colorbar import Colorbar


# Get main path of the repository and add it to the path:
try:
    # For scripts
    repo_path = Path(__file__).parent.parent.parent.resolve()
except NameError:
    # For Jupyter notebooks - add the known project root
    repo_path = Path.cwd().parent.parent.resolve()

# Add parent directory to path
sys.path.insert(0, str(repo_path))

from ml_events_utils import stylesheet_default
from ml_events_utils import color_gio as color_dict
from ml_events_utils import color_deep as color_dict

import matplotlib.pyplot as plt
import numpy as np

plt.style.use(stylesheet_default)

Observables:

- $\sigma_{\mathrm{tot}}$:     (NN/ POWHEG: `totxsec`,    RFR: ``)
- $p_{T, \, e^{+}}$:            (NN/ POWHEG: `ptep`,       RFR: `ptep`)
- $y_{e^{+}}$:                 (NN/ POWHEG: `yep`,        RFR: `yep`)
- $\cos(\theta^{*}_{e^{+}})$:  (NN/ POWHEG: `cthep`,      RFR: `cthep`)
- $m_{Z}$:                     (NN/ POWHEG: `invmass_Z1`, RFR: `mepem`)
- $\Delta \phi_{ee}$:          (NN/ POWHEG: `dphiee`,     RFR: `dphiee`)  # TODO: Adjust LaTex
- $p_{T, \, Z}$:                  (NN/ POWHEG: `ptee`,       RFR: ``)
- $p_{T, \, 4l}$:              (NN/ POWHEG: `pt4l`,       RFR: `pt4l`)
- $r_{\mathrm{LL}}$:           (NN:         `r_LL`,       RFR: `rll`)

In [35]:
observables = {
    "totxsec": {"NN": "totxsec",    "POWHEG": "totxsec",    "RFR": None},
    "ptep":    {"NN": "ptep",       "POWHEG": "ptep",       "RFR": "ptep"},
    "yep":     {"NN": "yep",        "POWHEG": "yep",        "RFR": "yep"},
    "cthep":   {"NN": "cthep",      "POWHEG": "cthep",      "RFR": "cthep"},
    "mepem":   {"NN": "invmass_Z1", "POWHEG": "invmass_Z1", "RFR": "mepem"},
    "dphiee":  {"NN": "dphiee",     "POWHEG": "dphiee",     "RFR": "dphiee"},
    "ptee":    {"NN": "ptee",       "POWHEG": "ptee",       "RFR": None},
    "pt4l":    {"NN": "pt4l",       "POWHEG": "pt4l",       "RFR": "pt4l"},
    "rll":     {"NN": "r_LL",       "POWHEG": None,         "RFR": "rll"},
}
print(f"Get observables like this: {observables['totxsec']['NN'] = }.")

observables_text = {
    "totxsec": r"$\sigma_{\mathrm{tot}}$",
    "ptep":    r"$p_{\mathrm{T}\, ee}$",
    "yep":     r"$y_{e^{+}}$",
    "cthep":   r"$\cos(\theta^{*}_{e^{+}})$",
    "mepem":   r"$m_{Z}$",
    "dphiee":  r"$\Delta \phi_{ee}$",  # TODO: Adjust LaTex.
    "ptee":    r"$p_{\mathrm{T} \, Z}$",
    "pt4l":    r"$p_{\mathrm{T} \, 4l}$",
    "rll":     r"$r_{\mathrm{LL}}$"
}

Get observables like this: observables['totxsec']['NN'] = 'totxsec'.


## Load the data

1. Put path of the analysis files in dictionary.

In [9]:
rfr_path = repo_path / "random_forest_regression"
nn_path  = None # repo_path / "neural_networks"
paths = {
    "NN": {
            "LO":    nn_path,
            "LOwS":  nn_path,
            "NLO":   nn_path,
            "NLOPS": nn_path,
           },
    "RFR": {
            "LO":    rfr_path / "histograms_rfr_ct_lo.top",
            "LOwS":  rfr_path / "histograms_rfr_ct_lows.top",
            "NLO":   rfr_path / "histograms_rfr_ct_nlo.top",
            "NLOPS": rfr_path / "histograms_rfr_ct_nlops.top",
           },
}

# Check that files exist:
for model, model_paths in paths.items():
    for order, path in model_paths.items():
        if path is not None and not path.exists():
            raise FileNotFoundError(f"File for {model} @ {order} does not exist: {path}")

print(f"Get paths like this: {paths['RFR']['LO'] = }.")

Get paths like this: paths['RFR']['LO'] = PosixPath('/ptmp/mpp/jlinder/ML_Giovanni/polarisation_tagging/random_forest_regression/histograms_rfr_ct_lo.top').


2. Provide a routine which reads in the histograms. Look for lines starting with, for example, `# dphiee` and ignore the possible `index <some number>` which would be there in a proper POWHEG histogram.

In [31]:
def read_histogram(path: Path, observable_map: Dict[str, str], model: str) -> Dict[str, Any]:
    """Read a histogram file and return a dictionary of histograms."""

    histograms = {}
    with open(path, "r") as f:
        in_histogram = False
        for line in f:
            line = line.strip()
            if not line and not in_histogram:
                continue

            if line.startswith("#") and not in_histogram:
                match = re.match(r"#\s*([a-zA-Z0-9_-]+)\s*", line)
                if not match:
                    continue
                observable = match.group(1)
                aligned_observable_name = observable_map.get(observable, observable)
                histograms[aligned_observable_name] = []
                in_histogram = True
                continue

            if in_histogram:
                for char in ["d", "D", "E"]:
                    line = re.sub(rf"{char}", "e", line)

                parts = line.split()
                if len(parts) != 4:
                    in_histogram = False
                    histograms[aligned_observable_name] = np.array(histograms[aligned_observable_name]).transpose()
                    continue

                values = np.array(list(map(float, parts[:])))
                histograms[aligned_observable_name].append(values)
    return histograms

3. Read in the histograms and align observable names.

In [33]:
histograms = {}
for model, model_paths in paths.items():
    histograms[model] = {}
    observable_map_model = {obs_info[model]: obs for obs, obs_info in observables.items()}
    for order, path in model_paths.items():
        if path is not None:
            print(f"Reading histogram for {model} @ {order} from {path}.")
            histograms[model][order] = read_histogram(path, observable_map=observable_map_model, model=model)
            print(f"Available observables for {model} @ {order}: {list(histograms[model][order].keys())}")

# test_histogram = read_histogram(paths["RFR"]["LO"], observable_map=observables, model="RFR")

print(f"Get histograms like this: {histograms['RFR']['LO']['ptep'] = }.")
print(f"Access left bin edges like this: {histograms['RFR']['LO']['ptep'][0] = }.")
print("Have fun plotting! :)")

Reading histogram for RFR @ LO from /ptmp/mpp/jlinder/ML_Giovanni/polarisation_tagging/random_forest_regression/histograms_rfr_ct_lo.top.
Available observables for RFR @ LO: ['yep', 'dphiee', 'cthep', 'rll', 'pt4l', 'ptep', 'mepem']
Reading histogram for RFR @ LOwS from /ptmp/mpp/jlinder/ML_Giovanni/polarisation_tagging/random_forest_regression/histograms_rfr_ct_lows.top.
Available observables for RFR @ LOwS: ['yep', 'dphiee', 'cthep', 'rll', 'pt4l', 'ptep', 'mepem']
Reading histogram for RFR @ NLO from /ptmp/mpp/jlinder/ML_Giovanni/polarisation_tagging/random_forest_regression/histograms_rfr_ct_nlo.top.
Available observables for RFR @ NLO: ['yep', 'dphiee', 'cthep', 'rll', 'pt4l', 'ptep', 'mepem']
Reading histogram for RFR @ NLOPS from /ptmp/mpp/jlinder/ML_Giovanni/polarisation_tagging/random_forest_regression/histograms_rfr_ct_nlops.top.
Available observables for RFR @ NLOPS: ['yep', 'dphiee', 'cthep', 'rll', 'pt4l', 'ptep', 'mepem']
Get histograms like this: histograms['RFR']['LO'][

## Plotting routine for ordinary kinematic data

In [ ]:
def plot_histograms(ax, weights_sig, labels, styles:list=[], uncertainties=False):
    """
    Plot histograms on the given axis.

    Parameters:
        ax: The axis to plot on.
        weights_sig: A list of tuples (bin_edges[0:n+1], weights[0:n], sigs_stat[0:n]) for each histogram to plot.
        labels: A list of labels for the histograms.
        styles: A list of style dictionaries for each histogram (e.g. color, linestyle).
                To change, for example, only the color of the second histogram, you can provide styles=[{}, {"color": "red"}, {}, ...].
        uncertainties: Whether to plot uncertainty bands.
    """
    if labels is None:
        labels = [None,] * len(weights_sig)
    if not styles:
        styles = [{},] * len(weights_sig)

    default_style = {"alpha": 1.0, "marker": "", "linestyle": "-"}  # Color is set by the color cycle or by the user-provided styles.
    for i, (bs, ws, sigs_stat) in enumerate(weights_sig):
        # Use step instead of hist to avoid edge lines at the sides
        style_settings = default_style.copy()
        style_settings.update(styles[i])
        line = ax.step(bs[:-1] + np.diff(bs)/2, ws, where='post', label=labels[i], **style_settings)

        # Ignore uncertainties for now.
        # if uncertainties:
        #     hist_colour    = line[0].get_color()
        #     hist_linestyle = line[0].get_linestyle()
        #     td.Data.uncertainty_bands(ax, bs, ws, sigs_stat, clr=hist_colour, ls=hist_linestyle, bands=True)

In [ ]:
def plot_mem_observable(histograms: dict,
                        xrange: list,
                        title:str,
                        xlabel:str,
                        unit:str,
                        ylabel1:str,
                        yranges:list=[],
                        logs:list = [False, False],
                        legends:list=[{}, {}],
                        rcParams:dict={},):

    """
    histogram: dict = {
                       "model1": histogram,
                       "model2": histogram,
                      }
    Plots:
        Oben:
        Ratio1:
        Ratio2:
    """

    with plt.style.context(stylesheet_default):
        mpl.rcParams.update(rcParams)
        size = mpl.rcParams['figure.figsize']
        n_plots = 3
        scale_factor = 1.0 + 0.5*(n_plots-1)
        height_ratios = [3,] + [1,]*(n_plots-1)
        fig, axs = plt.subplots(n_plots, 1, sharex=True, figsize=(size[0], scale_factor*size[1]), height_ratios=height_ratios)
        ylabels = ["",] * n_plots

        pass